# MEPS data pipeline

Run the cells from top to bottom to download the approved MEPS files, build the Panel 27 person-round history, and create the study cohort and discontinuation label. Raw data and generated tables stay in Git-ignored `data/` folders.

## Find the repository

This lets the notebook run when Jupyter starts in the repository root or in `notebooks/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "download_meps.py").is_file()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not find the repository root from the notebook directory")

print(f"Repository: {ROOT}")

## 1. Download the MEPS source files

The downloader skips existing archives that pass ZIP validation.

In [ ]:
subprocess.run(
    [sys.executable, str(ROOT / "src" / "download_meps.py")],
    cwd=ROOT,
    check=True,
)

## 2. Build the person-round history

Prescription and condition records are summarized before they are joined to the longitudinal data. The script writes the linked history, join audit files, and a deduplicated statin-purchase extract to `data/interim/`.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.build_person_round_history"],
    cwd=ROOT,
    check=True,
)

## 3. Build the study cohort and label

This uses the linked person-round history and writes the cohort, attrition flow, label balance, and quality summary to `data/processed/`.

In [ ]:
subprocess.run(
    [sys.executable, str(ROOT / "src" / "build_study_cohort.py")],
    cwd=ROOT,
    check=True,
)

## Review the outputs

The first table is one row per person-round. The statin extract is one row per unique statin purchase; the remaining tables summarize linkage, cohort inclusion, and the outcome class balance.

In [ ]:
import pandas as pd

history = pd.read_csv(
    ROOT / "data" / "interim" / "person_round_history.csv",
    dtype={"DUPERSID": "string"},
)
statin_purchases = pd.read_csv(
    ROOT / "data" / "interim" / "statin_purchases.csv",
    dtype={"DUPERSID": "string"},
)
join_audit = pd.read_csv(ROOT / "data" / "interim" / "join_audit.csv")
unmatched_keys = pd.read_csv(ROOT / "data" / "interim" / "unmatched_linkage_keys.csv")
cohort_flow = pd.read_csv(ROOT / "data" / "processed" / "cohort_flow.csv")
label_balance = pd.read_csv(ROOT / "data" / "processed" / "label_balance.csv")

print(f"Person-round rows: {len(history):,}")
print(f"Unique statin purchases: {len(statin_purchases):,}")
print(
    "Purchases without a standardized drug name: "
    f"{statin_purchases['standardized_statin'].fillna('').eq('').sum():,}"
)
print(statin_purchases.head(10).to_string(index=False))
print(join_audit.to_string(index=False))
print(f"Unmatched source summary keys: {len(unmatched_keys):,}")
print(cohort_flow.to_string(index=False))
print(label_balance.to_string(index=False))